# 05 · Behavioural features

Sender history **before** the current transaction. `sender_previous_fraud_rate` never includes the current label.

In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from cross_model_drift.data import load_transactions, write_parquet
from cross_model_drift.features import BEHAVIOURAL_FEATURES, engineer_features
from cross_model_drift.notebook import setup_model_session
from cross_model_drift.splits import V1_TRAIN

nb = setup_model_session()
raw = load_transactions(nb.config, engine=nb.engine, windows=[V1_TRAIN], limit=150_000)
feat = engineer_features(raw)
feat[list(BEHAVIOURAL_FEATURES)].head()

## Leakage check

First transaction per sender must have empty history.

In [ ]:
firsts = feat.sort_values(["sender_id", "created"]).groupby("sender_id", as_index=False).head(1)
assert (firsts["sender_tx_count_10m"] == 0).all()
assert (firsts["sender_previous_fraud_rate"] == 0).all()
firsts[["sender_id", "created", *BEHAVIOURAL_FEATURES]].head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.0))
sns.histplot(feat, x="sender_tx_count_24h", bins=30, ax=axes[0], color="#4C78A8")
axes[0].set_title("tx count 24h")
sns.histplot(feat, x="sender_avg_amount", bins=30, ax=axes[1], color="#F58518")
axes[1].set_title("prior avg amount")
sns.histplot(feat, x="sender_previous_fraud_rate", bins=20, ax=axes[2], color="#E45756")
axes[2].set_title("prior fraud rate")
nb.show(fig)

In [ ]:
corr = feat[list(BEHAVIOURAL_FEATURES) + ["is_fraud"]].corr(numeric_only=True)["is_fraud"].drop("is_fraud")
corr.sort_values().plot(kind="barh", color="#4C78A8", figsize=(8, 4.5))
plt.title("Correlation with is_fraud (sample)")
nb.show()

In [ ]:
path = nb.artifacts / "samples" / "v1_train_behavioural.parquet"
write_parquet(feat, path)
path, len(feat)